In [1]:
import sys
import subprocess
packages = ['sqlalchemy', 'psycopg[binary]', 'pgvector', 'langgraph', 'langchain-core', 'groq', 'pydantic', 'pandas', 'plotly', 'sentence-transformers', 'ipywidgets', 'pypdf', 'pytesseract', 'pillow', 'python-dotenv']
print(f"Installing into {sys.executable}...")
subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)
print("All packages installed successfully!")

Installing into C:\Users\baps\Desktop\hcl task\travel-reimbursement-agent\colab-env\Scripts\python.exe...
All packages installed successfully!


# Travel Reimbursement Approval Agent

## 1. Problem Statement
The goal is to build an intelligent, agentic AI system to process travel reimbursement claims. It must evaluate claims against company policy, calculate approved and deducted amounts deterministically, use Groq for reasoning, and store results (including embeddings) in PostgreSQL using `pgvector`.

## 2. Solution Overview
The system uses LangGraph to orchestrate a workflow that validates the claim, retrieves policy rules from PostgreSQL using pgvector, runs deterministic checkers (receipts, limits, airfare class, timeliness, approval thresholds), evaluates the claim using a strict policy engine, and finally calls Groq to provide human-readable reasoning.

## 3. Architecture
Claim Input → Validate Claim → Retrieve Relevant Policy → Run Tools → Deterministic Policy Engine → Groq Reasoning → Output Validation → Final Decision

## 4. Technology Stack
- Python 3.10+
- Groq (LLM)
- LangGraph (Orchestration)
- PostgreSQL + pgvector (Database & Embeddings)
- SQLAlchemy (ORM)
- Sentence-Transformers (Local Embeddings)
- Pydantic (Data validation)
- Plotly (Dashboard)

In [2]:
import sys
!{sys.executable} -m pip install -r requirements.txt


'C:\Users\baps\Desktop\hcl' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
!pip install pandas plotly
import sys
import os
sys.path.append(os.path.abspath('.'))
import json
import pandas as pd
import plotly.express as px
from src.database import DatabaseManager
from src.embeddings import EmbeddingProvider
from src.policy_repository import PolicyRepository
from src.groq_client import GroqLLMProvider
from src.agent import TravelReimbursementAgent


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 5. Environment Setup & 6. Database Setup

In [4]:
db_manager = DatabaseManager()
db_manager.setup_database()
print("Database and pgvector extension configured successfully.")

Database and pgvector extension configured successfully.


## 7. PostgreSQL + pgvector & 8. Policy Knowledge Base
**Important Note on pgvector**: The system dynamically stores and retrieves ALL policies via `pgvector` inside PostgreSQL. The `policy.json` file is strictly used *only once* below to seed the database initially. During actual execution, the LangGraph agent queries `pgvector` live to find policies.

In [5]:
def ingest_policy_rules():
    embedding_provider = EmbeddingProvider()
    policy_repo = PolicyRepository(db_manager, embedding_provider)
    count = policy_repo.ingest_policy_rules('data/policy.json')
    print(f"Ingested {count} new policy rules into pgvector.")
    return policy_repo, embedding_provider

policy_repo, embedding_provider = ingest_policy_rules()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ingested 0 new policy rules into pgvector.


## Interactive Bill Upload (Mock OCR)
Here you can upload a `.pdf` or `.png`/`.jpg` bill. This simulates how a real user would submit physical receipts, which our `extract_receipt_data` tool processes via OCR.

In [6]:
import json
from src.tools import extract_receipt_data
import os
import ipywidgets as widgets
from IPython.display import display
from src.groq_client import GroqLLMProvider
import re

# File Upload Widget
uploader = widgets.FileUpload(
    accept='.pdf, .png, .jpg',  # Accepted file extension
    multiple=False,
    description='Upload Bill'
)

# Text Area for manual JSON input
manual_input = widgets.Textarea(
    value='',
    placeholder='Paste claim JSON here or type a natural language sentence (e.g., "spend $500 for medical")...',
    description='Manual Input:',
    layout=widgets.Layout(width='400px', height='100px')
)

# Process Button for Manual Input
process_button = widgets.Button(
    description='Process Manual Input',
    button_style='info'
)

output_area = widgets.Output()

# Layout
ui_box = widgets.VBox([
    widgets.HBox([uploader, widgets.Label("  OR  "), manual_input, process_button]),
    output_area
])
display(ui_box)

def handle_results(result):
    global results
    if 'results' in globals():
        if not globals().get('static_cleared', False):
            results.clear()
            globals()['static_cleared'] = True
        results.append(result)
        if 'render_dashboard' in globals():
            render_dashboard()
        if 'render_audit_trail' in globals():
            render_audit_trail()
        if 'render_final_json' in globals():
            render_final_json()

def on_upload_change(change):
    if uploader.value:
        with output_area:
            output_area.clear_output()
            # Handle new dict format in latest ipywidgets
            if isinstance(uploader.value, dict) and len(uploader.value) > 0:
                first_key = list(uploader.value.keys())[0]
                uploaded_file = uploader.value[first_key]
                if isinstance(uploaded_file, dict) and 'content' in uploaded_file:
                    content = uploaded_file['content']
                    filename = uploaded_file.get('name', uploaded_file.get('metadata', {}).get('name'))
                else:
                    content = uploaded_file.content
                    filename = uploaded_file.name
            elif isinstance(uploader.value, tuple) and len(uploader.value) > 0:
                uploaded_file = uploader.value[0]
                content = uploaded_file['content']
                filename = uploaded_file.get('name', uploaded_file.get('metadata', {}).get('name'))
                
            os.makedirs('temp', exist_ok=True)
            filepath = os.path.join('temp', filename)
            with open(filepath, 'wb') as f:
                f.write(content)
            
            print(f"\nProcessing {filename}...")
            print("1. Extracting data via OCR/PDF Parsing...")
            extracted = extract_receipt_data(filepath)
            
            # Construct a full claim object from the extracted receipt
            new_claim = {
                "claim_id": "UPL-" + str(hash(filename))[-6:],
                "employee_id": "EMP-999",
                "items": [
                    {
                        "receipt_id": "REC-" + str(hash(filename))[-4:],
                        "amount": extracted.get("extracted_amount", 0.0),
                        "currency": "USD",
                        "date": extracted.get("extracted_date", "2024-01-01"),
                        "category": extracted.get("category", "Meals"),
                        "description": "Unknown",
                        "merchant": extracted.get("merchant", "Unknown"),
                        "receipt_attached": True
                    }
                ]
            }
            
            print("\n2. Extracted Data:")
            print(json.dumps(new_claim, indent=2))
            
            print("\n3. Running through LangGraph Agent...")
            try:
                result = agent.process_claim(new_claim)
                print(f"\n✅ FINAL DECISION: {result['decision']}")
                print(f"Approved: ${result['approved_amount']} | Deducted: ${result['deducted_amount']}")
                if result['missing_docs']:
                    print(f"Missing Docs: {result['missing_docs']}")
                print(f"Reasoning: {result['explanation']}")
                handle_results(result)
            except NameError:
                print("\n[Error] The LangGraph Agent is not initialized yet. Please scroll down and run the cells in Section 10 to 20 to initialize 'agent', then try uploading again!")

def parse_natural_language_claim(text):
    llm = GroqLLMProvider()
    sys_prompt = 'You extract claim data from natural language text. Output ONLY valid JSON representing the claim. The JSON MUST match this exact schema: {"claim_id": "TXT-123", "employee_id": "EMP-001", "items": [{"receipt_id": "REC-123", "amount": 500.0, "currency": "USD", "date": "2024-01-01", "category": "string", "description": "string", "merchant": "string", "receipt_attached": false}]}'
    response = llm.invoke(f"Extract details from: {text}", system_prompt=sys_prompt)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    match = re.search(r'```json\s*(.*?)\s*```', response, re.DOTALL)
    if match: return json.loads(match.group(1))
    
    start_idx = response.find('{')
    end_idx = response.rfind('}')
    if start_idx != -1 and end_idx != -1:
        return json.loads(response[start_idx:end_idx+1])
    return json.loads(response)

def on_manual_process(b):
    with output_area:
        output_area.clear_output()
        text_val = manual_input.value.strip()
        if not text_val:
            print("Please enter valid JSON or text.")
            return
            
        try:
            # First try parsing as JSON directly
            new_claim = json.loads(text_val)
            print("\nProcessing Manual JSON...")
        except json.JSONDecodeError:
            # If it's not JSON, assume it's natural language and parse with LLM!
            print("\nParsing natural language text with AI...")
            try:
                new_claim = parse_natural_language_claim(text_val)
                print("Successfully parsed into structured claim:")
            except Exception as e:
                print(f"Failed to parse text into claim: {str(e)}")
                return
                
        print(json.dumps(new_claim, indent=2))
        print("\nRunning through LangGraph Agent...")
        try:
            result = agent.process_claim(new_claim)
            print(f"\n✅ FINAL DECISION: {result['decision']}")
            print(f"Approved: ${result['approved_amount']} | Deducted: ${result['deducted_amount']}")
            if result['missing_docs']:
                print(f"Missing Docs: {result['missing_docs']}")
            print(f"Reasoning: {result['explanation']}")
            handle_results(result)
        except NameError:
            print("\n[Error] The LangGraph Agent is not initialized yet. Please scroll down and run the cells in Section 10 to 20 to initialize 'agent'!")

uploader.observe(on_upload_change, names='value')
process_button.on_click(on_manual_process)





## 10. Data Models to 20. LangGraph Agent
The core logic has been encapsulated in the `src/` modules for readability and reusability.

In [7]:
llm_provider = GroqLLMProvider()
agent = TravelReimbursementAgent(policy_repo, llm_provider, db_manager)

## 22. Evaluate Five Claims
We will load the exactly provided 5 claims from `data/claims.json` and process them through our agent.

In [8]:
with open('data/claims.json', 'r') as f:
    claims = json.load(f)

results = []

for claim in claims:
    result = agent.process_claim(claim)
    results.append(result)

print(f"Processed {len(results)} claims successfully.")



Processed 6 claims successfully.


## 21. Audit Trail
Let's fetch the audit logs from PostgreSQL to show the execution details.

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

audit_output = widgets.Output()
display(audit_output)

def render_audit_trail():
    with audit_output:
        clear_output(wait=True)
        session = db_manager.get_session()
        try:
            from src.database import AuditLog
            logs = session.query(AuditLog).all()
            log_data = [{
                "timestamp": log.timestamp,
                "claim_id": log.claim_id,
                "agent_node": log.agent_node,
                "tool_name": log.tool_name
            } for log in logs]
            if log_data:
                df_logs = pd.DataFrame(log_data)
                display(df_logs.tail(10))
            else:
                print("No audit logs found yet.")
        finally:
            session.close()

render_audit_trail()



Output()

## Dashboard
Using Plotly to show decision distribution and financial totals.

In [10]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

dashboard_output = widgets.Output()
display(dashboard_output)

def render_dashboard():
    with dashboard_output:
        clear_output(wait=True)
        df = pd.DataFrame(results)
        if df.empty:
            print("📊 Dashboard is waiting... Please process a claim using the UI above!")
        else:
            # 1. Decision distribution
            fig1 = px.pie(df, names='decision', title='Decision Distribution')
            fig1.show()

            # 2. Approved vs Deducted Amount
            amounts = pd.DataFrame({
                'Type': ['Approved', 'Deducted'],
                'Total': [df['approved_amount'].sum(), df['deducted_amount'].sum()]
            })
            fig2 = px.bar(amounts, x='Type', y='Total', title='Financial Totals', color='Type')
            fig2.show()

            display(df[['claim_id', 'decision', 'approved_amount', 'deducted_amount']])

# Initial render
render_dashboard()



Output()

## 24. Testing
Verifying the system independently derived the exact expected results.

In [11]:
expected_decisions = {
    "CLM-001": "APPROVE",
    "CLM-002": "REJECT",
    "CLM-003": "PARTIAL_APPROVE",
    "CLM-004": "MANUAL_REVIEW",
    "CLM-005": "MANUAL_REVIEW",
    "CLM-006": "MANUAL_REVIEW"
}
# 
for r in results:
    if r['claim_id'].startswith('CLM-'):
        expected = expected_decisions.get(r['claim_id'], "UNKNOWN")
        assert r['decision'] == expected, f"Failed for {r['claim_id']}: Expected {expected}, got {r['decision']}"
# 
print("All assertions passed successfully! The deterministic engine works as expected.")
# 



All assertions passed successfully! The deterministic engine works as expected.


## 25. Design Notes & Reasoning
**Why Groq:** Fast inference, suitable for agent reasoning, and simple API integration.
**Why PostgreSQL:** Structured persistence, familiar relational database, supports claims, items, policies and audit logs simultaneously.
**Why pgvector:** Avoids separate vector database, policy corpus is small, combines relational data and vector search, simpler deployment, and easier auditability.
**Why LangGraph:** Explicit agent workflow, conditional routing, state management, easier debugging, and tool execution visibility.
**Why deterministic policy engine:** Financial correctness, reproducibility, and prevents LLM arithmetic errors.
**Why Manual Review:** Ambiguity should not be hallucinated; human oversight is required for exceptions.

## 26. Assumptions & Limitations
**Assumptions:**
- Assignment currency is USD.
- Policy is treated as authoritative.
- The provided receipt metadata is trusted.
- Manual-review cases do not receive fabricated reimbursement amounts.
- Confidence is a heuristic, not a calibrated probability.

**Limitations:**
- Mock receipts rather than actual OCR.
- Small policy corpus.
- No production authentication or real approval workflow.

## 27. Future Improvements
- OCR receipt extraction and image understanding.
- Fraud detection and duplicate receipt detection.
- Policy versioning and human approval portal.
- Role-based access and Langfuse observability.

## 28. Interview Explanation
### How I Explain This Project
1. **Problem**: Automating travel reimbursements with high accuracy while keeping a human in the loop for ambiguity.
2. **Architecture**: A pipeline starting with data validation, moving into RAG for policy context, tool execution for rules, a deterministic engine for finances, and finally an LLM for reasoning.
3. **PostgreSQL + pgvector**: Unified database for structured data (claims, audits) and vector search (policies), avoiding the operational overhead of a separate vector DB.
4. **Policy RAG & LangGraph**: LangGraph provides explicit step-by-step state management. RAG ensures the LLM reasons based on exact company policy.
5. **Deterministic Engine**: "Why didn't you let the LLM calculate the reimbursement?" Because financial calculations and policy enforcement should be deterministic and auditable. I use the LLM for reasoning and explanation, while Python functions enforce the actual monetary rules.
6. **Structured Output & Dashboard**: Ensures downstream systems can safely consume the data, and provides visibility to stakeholders.

## 29. Final JSON Results

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

final_json_output = widgets.Output()
display(final_json_output)

def render_final_json():
    with final_json_output:
        clear_output(wait=True)
        print(json.dumps(results, indent=2))

render_final_json()



Output()